In [ ]:
!pip install seaborn matplotlib
#import seaborn as sns
#sns.__version__

In [ ]:
!pip install -U pyALE shap -q


## **Feature interactions on tangerines**

Hi, everyone! This practice was prepared over the New Year holidays. In it we will work with a dataset about tangerines and look at the features of this wonderful citrus from every angle!

P.S I hope you are not allergic to tangerines!
Happy coding! 🤗

### **Problem statement**
Let us look at the problem of predicting the quality of tangerines. In terms of the statement, quality is a continuous variable in the range from 0 to 5. The intermediate values `1.5, ..., 4.5` are meaningful too, and the quality scores present in the dataset, although discrete with a step of 0.5, are imbalanced. That is why we try to solve a regression problem.

In [ ]:
import pandas as pd
import numpy as np
from PyALE import ale

import matplotlib.pyplot as plt
from sklearn.inspection import PartialDependenceDisplay, partial_dependence

from shap import TreeExplainer

from sklearn.tree import plot_tree


LINK = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/orange_dataset.csv'
data = pd.read_csv(LINK, index_col=0)

### **Building the model**

In the cell below we will simply train a model using `MinMaxScaler`. The transformation (scaling) is done like this:

1. We choose the desired range of the features — usually $(0, 1)$. Let us denote $x_{min} = 0, x_{max} = 1$:

2. We scale the data by the formula:

$$X_{scaled} = \frac{X-min(X)}{max(X) - min(X)}*(x_{max} - x_{min} + x_{min})$$

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Preparing the dataset for training
X = data.drop('Quality (1-5)', axis=1)
y = data['Quality (1-5)']

feature_names = list(data.columns)
feature_names.remove('Quality (1-5)')

# Splitting into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Scaling
sc = MinMaxScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

X_train = pd.DataFrame(X_train, columns = feature_names)
X_test = pd.DataFrame(X_test, columns = feature_names)

As the model we will use a decision tree.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Training the model
tree = DecisionTreeRegressor(random_state=42, max_depth=4)
tree.fit(X_train, y_train)

#predictions = np.clip(tree.predict(X_test), 1, 5)
predictions = tree.predict(X_test)

mse = mean_squared_error(y_test, predictions)
print(f'Model MSE : {mse}, rmse: {mse**(1/2)}')

**Note:** We could clip the prediction so that it does not go outside the ranges. But a tree, by construction, will not go outside the range it was trained on.

**Task 1.**

Make a prediction on the test data `X_test`. What is the maximum value of the prediction?

Round the answer to two decimal places.

In [ ]:
# Your code here

### **Building importance maps**

Let us pick a couple of features for the analysis, based on their meaning in the problem. Among the features we have:

- `Size (cm)` — the size of the tangerine in centimetres.  
- `Weight (g)` — the weight of the tangerine in grams.  
- `Brix (Sweetness)` — the sweetness score of the tangerine
- `pH (Acidity)` — the acidity level of the tangerine.  
- `Softness (1-5)` — the softness of the tangerine on a scale from 1 (very hard) to 5 (very soft).  
- `HarvestTime (days)` — the harvest time, given in days since the beginning of the season.  
- `Ripeness (1-5)` — the degree of ripeness of the tangerine on a scale from 1 (unripe) to 5 (fully ripe).  
- `Blemishes (Y/N)` — the presence of damage or defects on the tangerine peel (Y — yes, N — no).  
- `Color_Light Orange` — the colour of the tangerine: light orange.  
- `Color_Orange` — the colour of the tangerine: orange.  
- `Color_Orange-Red` — the colour of the tangerine: orange-red.  
- `Color_Yellow-Orange` — the colour of the tangerine: yellow-orange.  

**Task 2.**

How many binary features are there in the data?

`Your answer here.`

### **Sweetness and acidity**
Let us choose the features for the analysis. Let us look at how sweetness `Brix (Sweetness)` and acidity `pH (Acidity)` influence the prediction. To begin with, we will look at the one-dimensional influence using **PDP** and **ALE** plots.

In [ ]:
feature_idx = 2
feature_name = data.columns[feature_idx]

print(f'Analysing {feature_name}')

In [ ]:
results = partial_dependence(tree, X_test, [feature_idx], grid_resolution=20, kind='average')
#results = partial_dependence(tree, X_test, [0], grid_resolution=20, kind='individual') == ICE

results
plt.plot(results['grid_values'][0], results['average'][0])

plt.title(f'PDP plot for {feature_name}')
plt.xlabel(f'{feature_name} value')
plt.ylabel('model prediction');

You can see that there is a certain sweetness threshold, above which the model tends to assign a better quality to the tangerine. And what about acidity?

**Task 3.**
Build a one-dimensional plot for acidity and draw a conclusion about the nature of the influence.

In [ ]:
feature_idx = # Your code here
feature_name = # Your code here

print(f'Analysing {feature_name}')

In [ ]:
# Your code here

results =

plt.plot(results['grid_values'][0], results['average'][0])

plt.title(f'PDP plot for {feature_name}')
plt.xlabel(f'{feature_name} value')
plt.ylabel('model prediction');

If we look at ALE, the dependence will be similar. You can check this by building the ALE for ph yourself.

In [ ]:
brix = 'Brix (Sweetness)'
ph = 'pH (Acidity)'

ale_eff = ale(
    X=pd.DataFrame(X_test, columns=feature_names), model=tree, feature=[brix], grid_size=50, include_CI=True
)

In [ ]:
# Your code here

### **Assessing the two-dimensional influence**

Let us move on to assessing the features in pairs. We will look at the joint influence of `Brix (Sweetness)` and `pH (Acidity)` in two ways.

In [ ]:
# Let us build a pdp plot
PartialDependenceDisplay.from_estimator(tree, X_test, [(2, 3)])
plt.title('PDP plot for interactions');

**Task 4.**
Analyse the plot above. In the same way, build a two-dimensional plot for `Brix` paired with the features `Size, Weight, Harvest Time`.  

Which of these features can we put forward a hypothesis about, regarding a joint influence with `Brix`?

In [ ]:
# Your code here

`Your answer here.`

**Task 5.**

For the feature from task 4, build the ALE. What happens to the joint effect if we correct it for the intervals?

In [ ]:
# hint: use impute_empty_cells

# Your code here

### **SHAP**

Obtaining interaction values in shap repeats the steps of building this method of analysing feature contributions, for the case where we compute an individual contribution.

In [ ]:
pred = tree.predict(X_test)

explainer = TreeExplainer(tree)
explanation = explainer(X_test)

**Task 6.**

Using the attributes of the explainer, pull out the interaction values. `dir(explainer)` will help you. The attribute contains the word `interaction` in its name and has no underscores. How many dimensions does the resulting object have?

In [ ]:
# Your code here

Compare this with the dimensionality of `X_test`. The pairwise interaction is stored for every object in the data.

Let us analyse the pairwise interaction for a single object. We will fix the object with index 0 and look at the pairwise effects for it.

In [ ]:
pd.DataFrame(shap_interaction[0],index=feature_names,columns=feature_names)

**Task 7.** With which feature by type (apart from itself) does `Brix` have the strongest interaction for the object with index 0?

`Your answer here`

In [ ]:
#Let us compute the base value (the mean prediction)
mean_pred = np.mean(pred)

sum_shap = np.sum(shap_interaction[0])

print(f"Model prediction: {pred[0]}")
print(f"Mean prediction: {mean_pred}, shap sum: {sum_shap}")

There you go: the sum of all the interactions gives exactly the value added to the mean of the prediction.

It is more informative to analyse the interactions for all of them at once, in the form of a matrix. For that it is enough to average the resulting output over the objects (the zeroth axis) and build a heatmap.

In [ ]:
import seaborn as sns
# Averaging (you can take the median — it is more robust to values that are "too large" or "too small" among all the values — just in case)
mean_shap = np.round(np.median(np.abs(shap_interaction), 0), 4)

df = pd.DataFrame(mean_shap,index=feature_names,columns=feature_names)

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(df, ax=ax, annot=df)

plt.title('SHAP heatmap')
plt.xticks(rotation=45);

**Task 8.** With which feature (apart from itself) does `Brix` interact the most on average over the dataset?

`Your answer here.`

### **H-statistic**

Finally, let us use the H-statistic. Check whether the result agrees with shap?

In [ ]:
from itertools import combinations
from sklearn.inspection import partial_dependence

def centered_pd(model, X, features, grid_resolution=20):
    """Centred PD function: the mean over the grid has been subtracted from the values."""
    pd_res = partial_dependence(model, X, features=features,
                                grid_resolution=grid_resolution, kind="average")
    values = pd_res["average"][0]
    return values - values.mean(), pd_res["grid_values"]

def h_statistic(model, X, feat_i, feat_j, grid_resolution=20, eps=1e-3):
    """Friedman's H-statistic for a pair of features — straight from the formula in the lesson.

    H^2 = sum [PD_kj - PD_k - PD_j]^2 / sum PD_kj^2

    All the PD functions are centred, otherwise the difference does not vanish.
    If the denominator is close to zero (the joint effect of the pair is almost absent),
    the ratio is unstable and easily produces values like 9 or 16 — that is the noise
    of numerical division, not a strong interaction. We mark such pairs as NaN.
    """
    cols = list(X.columns)
    i, j = cols.index(feat_i), cols.index(feat_j)

    pd_ij, _ = centered_pd(model, X, [(i, j)], grid_resolution)
    pd_i, _ = centered_pd(model, X, [i], grid_resolution)
    pd_j, _ = centered_pd(model, X, [j], grid_resolution)

    numerator = ((pd_ij - pd_i[:, None] - pd_j[None, :]) ** 2).sum()
    denominator = (pd_ij ** 2).sum()
    if denominator < eps:            # the pair has almost no influence on the prediction
        return float("nan")
    return numerator / denominator

# we compute H for all the pairs of features
pairs = list(combinations(X_test.columns, 2))
h_values = {f"{a} — {b}": h_statistic(tree, X_test, a, b) for a, b in pairs}
h_series = pd.Series(h_values).dropna().sort_values(ascending=False)

print(f"pairs in total: {len(pairs)}, with a defined H: {len(h_series)}")
h_series.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
h_series.head(12).plot.barh(ax=ax, color="#2a78d6")
ax.invert_yaxis()
ax.set_xlabel("$H^2$")
ax.set_title("Friedman's H-statistic by pairs of features")
plt.tight_layout();

Note that the H-statistic suggests more interactions than shap. How true is that? It has to be checked in the model. In practice, Shapley values are more stable.

As an exercise — look at the code below and at the structure of the tree in it. The results we obtained are largely determined by the structure of the model, and it would have been quicker to find them by pulling out the structure of the tree. However, in the case of more complex algorithms, plots will prove to be an excellent way to avoid analysing a complex structure. Just try rerunning the notebook with a Random Forest.

In [ ]:
fig, ax = plt.subplots(figsize=(25, 10))
plot_tree(tree, proportion=True, ax=ax, feature_names=feature_names);